# 08 — Participant round: basic distributions

What the participant round contains, as figures. Run it after the round
closes; it is the same notebook that was checked on the pilot
(`07_pilot_distributions`). Every panel says "no data yet" rather than failing
while the round is empty. The propositions are in `09_study_propositions`.

Descriptive only: counts and distributions, no tests. Every figure states its
n. Sessions are pseudonymous (`S1, …` in consent order).

**Privacy.** This notebook reads participant rows. Outputs are stripped on
commit (`notebooks/study/.gitattributes`). Nothing prints a participant's
words, an id or an email. Figures go to `figures/<run_type>/`, which is
git-ignored. With few participants, a demographic figure can identify someone:
check before copying one into the thesis.

In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().resolve().parents[1]))

from collections import Counter

import matplotlib.pyplot as plt
import pandas as pd

from src import analysis, db, figures, pilot

# --- parameters -------------------------------------------------------------
RUN_TYPE = "study"
save = figures.Saver(RUN_TYPE, enabled=True)
figures.setup()
C = figures.ROUND_COLOR["participant"]

client = db.admin()
data = pilot.load(client, RUN_TYPE)
sessions = pilot.session_table(data)
consented = [s for s in sessions if s["status"] == "consented"]
turns = [t for t in data.suggestions]
dist = pilot.distributions(data)
print(f"run_type={RUN_TYPE!r}: {len(sessions)} sessions ({len(consented)} consented), {len(turns)} agent turns")

## 1. Sessions

In [ ]:
import statistics

fig, axs = plt.subplots(1, 3, figsize=(10, 2.8))
# One bar per session, unlabelled: the pseudonyms crowd the axis and say nothing a reader needs.
figures.bar_counts(axs[0], {s["session"]: s["turns"] for s in consented}, "Turns per session", "agent turns; line = median", color=C)
axs[0].set_yticks([]); axs[0].set_ylabel("sessions")
if consented:
    axs[0].axvline(statistics.median(s["turns"] for s in consented), color=figures.INK, linewidth=1.5, linestyle="--", zorder=3)
# First visit only: a return on a later day is calendar time, not time on the task (pilot.VISIT_GAP).
figures.strip(axs[1], [s["minutes first visit"] for s in consented if s["minutes first visit"] is not None],
              "Session span", "minutes, first to last turn", color=C)
figures.bar_counts(axs[2], Counter(s["status"] for s in sessions), "Consent", "sessions", color=C)
fig.tight_layout(); save(fig, "dist1_sessions"); plt.show()

In [ ]:
from datetime import datetime
for s in consented:
    sid = next(k for k, v in data.pseudonyms().items() if v == s["session"])
    ts = sorted(datetime.fromisoformat(x["occurred_at"]) for x in data.suggestions if x["session_id"] == sid)
    gaps = [round((b - a).total_seconds() / 60) for a, b in zip(ts, ts[1:])]
    if gaps and max(gaps) > 60:
        print(s["session"], len(ts), "turns, gaps (min):", gaps)

## 2. Requests

`request_type` is the app's label (D09). The shape is `shapes7` (D99), per
turn. A slash command is what the member typed first, if anything.

In [ ]:
fig, axs = plt.subplots(1, 3, figsize=(10, 3))
figures.bar_counts(axs[0], dist["request_type"], "Request type", "agent turns", color=C)
figures.bar_counts(axs[1], Counter(analysis.turn_shape(t) for t in turns), "Shape (shapes7)", "agent turns", color=C)
figures.bar_counts(axs[2], dist["command used"], "Slash command", "agent turns", color=C)
fig.tight_layout(); save(fig, "dist2_requests"); plt.show()

## 3. Tools

In [ ]:
fig, ax = plt.subplots(figsize=(6.4, 0.35 * max(len(dist["tool"]), 1) + 1))
figures.bar_counts(ax, dist["tool"], "Tool calls", "calls", color=C)
save(fig, "dist3_tools"); plt.show()

## 4. Performance per turn

Latency, model calls and tokens for every completed turn. Two measures on
different scales are never on one axis, so each has its own panel.

In [ ]:
done = [t for t in turns if t.get("completed_at")]
pick = lambda key, scale=1: [t[key] / scale for t in done if t.get(key) is not None]
fig, axs = plt.subplots(2, 2, figsize=(10, 4))
figures.strip(axs[0][0], pick("latency_ms", 1000), "Latency", "seconds", color=C)
figures.strip(axs[0][1], pick("llm_calls"), "Model calls", "calls per turn", color=C)
figures.strip(axs[1][0], pick("tokens_input", 1000), "Input tokens", "thousand tokens per turn", color=C)
figures.strip(axs[1][1], pick("tokens_output"), "Output tokens", "tokens per turn", color=C)
fig.tight_layout(); save(fig, "dist4_performance"); plt.show()

## 5. Feedback

`user_action` per turn that **offered something to accept** — a draft plan or
place cards (`analysis.actionable`). D17 stamps `ignored` on every turn with no
action after 24 hours, including advisory and factual replies that offered
nothing and turns that failed without a reply; counting those would read as
members ignoring suggestions. The excluded turns are counted below the figure.

Beside it: thumbs per reply, and the per-target action rows behind `user_action`.

In [ ]:
fig, axs = plt.subplots(1, 3, figsize=(10, 2.8))
from src.execution import _place_ids_of
replies = [t["reply_message_id"] for t in turns if t.get("reply_message_id")]
reply_refs = {m["id"]: _place_ids_of(m.get("metadata")) for m in (
    client.table("chat_messages").select("id, metadata").in_("id", replies).execute().data if replies else [])}
offered = [t for t in turns if analysis.actionable(t, reply_refs.get(t.get("reply_message_id"), ()))]
figures.bar_counts(axs[0], Counter(t.get("user_action") or "(not yet)" for t in offered),
                   "Reaction to a choice", "agent turns", color=C)
figures.bar_counts(axs[1], Counter(f.get("value") for f in data.feedback if f.get("value")), "Thumbs", "replies", color=C,
                   order=["up", "down"])
figures.bar_counts(axs[2], {f"{k} {a}": v for (k, a), v in dist["action rows"].items()}, "Actions on targets",
                   "rows", color=C)
fig.tight_layout(); save(fig, "dist5_feedback"); plt.show()
left_out = Counter("failed without a reply" if analysis.failed(t) else "offered nothing to accept"
                   for t in turns if t not in offered)
print(f"reaction panel: {len(offered)} of {len(turns)} turns; left out: {dict(left_out)}")

## 6. Participants

From the demographic questionnaire (D18). With a small n these can identify
someone; they are for checking, not for printing as they stand.

In [ ]:
# Stored codes (migration 20260918090000_study_instrumentation) → display text (D18), lowest first.
EDUCATION = {"compulsory": "compulsory", "vocational": "apprenticeship or vocational",
             "high_school": "matura or high school", "bachelor": "bachelor", "master_or_higher": "master or higher"}
GENDER = {"woman": "woman", "man": "man", "non_binary": "non-binary", "prefer_not_to_say": "prefer not to say"}
FAMILIARITY = ["never", "rarely", "monthly", "weekly", "daily"]

demo = data.demographics
fig, axs = plt.subplots(2, 2, figsize=(10, 4.4))
figures.strip(axs[0][0], [d["age"] for d in demo if d.get("age") is not None], "Age", "years", color=C)
figures.bar_counts(axs[0][1], Counter(d.get("gender") for d in demo if d.get("gender")), "Gender", "participants",
                   color=C, order=GENDER, labels=GENDER)
figures.bar_counts(axs[1][0], Counter(d.get("education_level") for d in demo if d.get("education_level")), "Education",
                   "participants", color=C, order=EDUCATION, labels=EDUCATION)
figures.bar_counts(axs[1][1], Counter(d.get("llm_familiarity") for d in demo if d.get("llm_familiarity")),
                   "LLM use", "participants", color=C, order=FAMILIARITY)
fig.tight_layout(); save(fig, "dist6_participants"); plt.show()

## 7. Post-task survey

UMUX-LITE through the UMUX-LITE → SUS conversion (an estimated SUS score)
against the Sauro–Lewis curved grading scale, the recommend item on 1–7, and
whether the member planned a real trip.

In [ ]:
usab = analysis.usability(data.post_task)
submitted = [p for p in data.post_task if p.get("submitted_at")]
fig = plt.figure(figsize=(10, 4))
grid = fig.add_gridspec(2, 2, height_ratios=[1, 1.3])
figures.grade_band(fig.add_subplot(grid[0, :]), [u["sus_equivalent"] for u in usab], analysis.CURVED_GRADES,
                   "Usability", "SUS-equivalent score (from UMUX-LITE)", color=C)
figures.bar_counts(fig.add_subplot(grid[1, 0]),
                   Counter(str(p["likelihood_to_recommend"]) for p in submitted if p.get("likelihood_to_recommend") is not None),
                   "Likelihood to recommend", "participants", color=C, order=[str(i) for i in range(1, 8)])
figures.bar_counts(fig.add_subplot(grid[1, 1]),
                   Counter({True: "real trip", False: "not a real trip"}.get(p.get("real_trip"), "no answer") for p in submitted),
                   "Real trip", "participants", color=C)
fig.tight_layout(); save(fig, "dist7_post_task"); plt.show()
print(f"post-task: {len(submitted)} submitted, {sum(1 for p in data.post_task if p.get('dismissed_at'))} dismissed")

## 8. Medians for the text

The median each strip and bar figure above draws, with its n, so the thesis can
name the value rather than read it off a plot. Same filters as the figures.
UMUX-LITE also as a mean with SD, the way it is reported and graded.
Aggregates only — nothing here prints a row.

In [ ]:
med = lambda xs: (statistics.median(xs), len(xs)) if xs else (None, 0)
rows = {
    "turns per session": med([s["turns"] for s in consented]),
    "session span, first visit (min)": med([s["minutes first visit"] for s in consented if s["minutes first visit"] is not None]),
    "latency (s)": med(pick("latency_ms", 1000)),
    "model calls per turn": med(pick("llm_calls")),
    "input tokens per turn": med(pick("tokens_input")),
    "output tokens per turn": med(pick("tokens_output")),
    "age (years)": med([d["age"] for d in demo if d.get("age") is not None]),
    "SUS-equivalent (UMUX-LITE)": med([u["sus_equivalent"] for u in usab]),
    "likelihood to recommend (1-7)": med([p["likelihood_to_recommend"] for p in submitted if p.get("likelihood_to_recommend") is not None]),
}
for name, (m, n) in rows.items():
    print(f"{name:34} median = {'—' if m is None else round(m, 2):>10}   (n = {n})")

# UMUX-LITE is conventionally reported as a mean, not a median (Lewis et al.; Sauro–Lewis grades a mean).
for key, label in [("umux_lite", "UMUX-LITE (0-100)"), ("sus_equivalent", "SUS-equivalent")]:
    xs = [u[key] for u in usab]
    if xs:
        sd = statistics.stdev(xs) if len(xs) > 1 else 0.0
        print(f"{label:34} mean   = {statistics.mean(xs):>10.2f}   SD = {sd:.2f}   (n = {len(xs)})")
if usab:
    print(f"{'grade of the mean SUS-equivalent':34}          {analysis.curved_grade(statistics.mean(u['sus_equivalent'] for u in usab))}")

In [ ]:
print(f"figures written to {save.dir.resolve()}")